In [ ]:
import json
import os
import pickle
from pathlib import Path

import pandas as pd
from anthropic import Anthropic
from anthropic.types.message_create_params import (
    MessageCreateParamsNonStreaming,
)
from anthropic.types.messages.batch_create_params import (
    Request,
)
from dotenv import load_dotenv
from openai import OpenAI
from utils import header_completion_test, row_completion_test, row_completion_test_2

## Settings

In [ ]:
# Read .env file
load_dotenv(
    dotenv_path=Path().resolve().parent.parent / ".env",
    override=False,
)

# Set OpenAI API key
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

# Set Anthropic API key
ANTHROPIC_API_KEY = os.environ["ANTHROPIC_API_KEY"]

In [ ]:
# Set directories
DATASET_DIR = Path("./dataset")
FEW_SHOT_DATASET_DIR = Path("./few_shot_dataset")

# Create directory for batch jsonl files
TIME_TAG = pd.Timestamp.now().strftime("%Y%m%d%H%M%S")
BATCH_INPUT_DIR = Path("./batch_input") / f"batch_input_{TIME_TAG}"
BATCH_INPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Model lists
OPENAI_NON_REASONING_MODELS = [
    "gpt-4.1-mini-2025-04-14",
    "gpt-4o-mini-2024-07-18",
]
OPENAI_REASONING_MODELS = [
    "gpt-5-mini-2025-08-07",
    "o4-mini-2025-04-16",
]
OPENAI_MODELS = OPENAI_NON_REASONING_MODELS + OPENAI_REASONING_MODELS

ANTHROPIC_MODELS = [
    "claude-sonnet-4-5-20250929",
    "claude-haiku-4-5-20251001",
    "claude-3-5-haiku-20241022",
]

In [ ]:
# Settings
DATASET_NAME_LIST = [
    "iris",
    "BH_1",
    "DA",
    "alkox",
    "oer_plate_a",
    "p3ht",
    "photo_pce10",
    "photo_wf3",
    "suzuki_edbo",
    "suzuki",
]
MODEL_LIST = ["gpt-4o-mini-2024-07-18"]
TEST_MODE = False  # True or False

COMMON_MAX_OUTPUT_TOKENS = 20000  # 8192 for claude-3-5-haiku-20241022, 20000 for others
COMMON_REASONING_EFFORT = "medium"  # "minimal", "low", "medium", "high"
COMMON_TEMPERATURE = 0.0

COMPLETION_MODE_LIST = ["header", "header2", "row", "row2"]

## Functions for batch processing

In [ ]:
# Function to generate OpenAI batch input file
def generate_openai_batch_input_file(
    dataset_name,
    model,
    completion_mode,
    max_output_tokens,
    reasoning_effort,
    temperature,
):
    # Read data
    dataset = pd.read_csv(DATASET_DIR / f"dataset_{dataset_name}.csv")

    with open(
        BATCH_INPUT_DIR / f"batch_input_{dataset_name}_{model}_{completion_mode}.jsonl", "w", encoding="utf-8"
    ) as f:
        if completion_mode == "header":
            settings = header_completion_test(dataset, few_shot_dataset_dir=FEW_SHOT_DATASET_DIR)
        elif completion_mode == "header2":
            settings = header_completion_test(dataset, few_shot_dataset_dir=FEW_SHOT_DATASET_DIR, wo_header=True)
        elif completion_mode == "row":
            settings = row_completion_test(dataset)
        elif completion_mode == "row2":
            settings = row_completion_test_2(dataset, few_shot_dataset_dir=FEW_SHOT_DATASET_DIR)
        else:
            raise ValueError(f"Unsupported completion mode: {completion_mode}")

        for id, setting in settings.items():
            system_prompt = setting["system_prompt"]
            user_prompts = setting["user_prompts"]
            user_prompts = [{"role": "system", "content": system_prompt}] + user_prompts

            # Generate request
            request = {
                "custom_id": f"request-{id}",
                "method": "POST",
                "url": "/v1/responses",
                "body": {
                    "model": model,
                    "input": user_prompts,
                },
            }
            if max_output_tokens is not None:
                request["body"]["max_output_tokens"] = max_output_tokens
            if reasoning_effort is not None:
                request["body"]["reasoning"] = {"effort": reasoning_effort}
            if temperature is not None:
                request["body"]["temperature"] = temperature

            f.write(json.dumps(request, ensure_ascii=False) + "\n")

    return settings

In [ ]:
# Function to generate Anthropic batch requests
def generate_anthropic_requests(
    dataset_name,
    model,
    completion_mode,
    max_output_tokens,
    temperature,
):
    # Read data
    dataset = pd.read_csv(DATASET_DIR / f"dataset_{dataset_name}.csv")

    if completion_mode == "header":
        settings = header_completion_test(dataset, few_shot_dataset_dir=FEW_SHOT_DATASET_DIR)
    elif completion_mode == "header2":
        settings = header_completion_test(dataset, few_shot_dataset_dir=FEW_SHOT_DATASET_DIR, wo_header=True)
    elif completion_mode == "row":
        settings = row_completion_test(dataset)
    elif completion_mode == "row2":
        settings = row_completion_test_2(dataset, few_shot_dataset_dir=FEW_SHOT_DATASET_DIR)
    else:
        raise ValueError(f"Unsupported completion mode: {completion_mode}")

    requests = []
    for id, setting in settings.items():
        system_prompt = setting["system_prompt"]
        user_prompts = setting["user_prompts"]

        # Generate request
        request = Request(
            custom_id=f"request-{id}",
            params=MessageCreateParamsNonStreaming(
                model=model,
                max_tokens=max_output_tokens,
                temperature=temperature,
                system=system_prompt,
                messages=user_prompts,
            ),
        )
        requests.append(request)

    # Save to jsonl file
    with open(
        BATCH_INPUT_DIR / f"batch_input_{dataset_name}_{model}_{completion_mode}.jsonl", "w", encoding="utf-8"
    ) as f:
        for request in requests:
            f.write(json.dumps(request, ensure_ascii=False) + "\n")

    return requests, settings

## Batch processing

In [ ]:
# Batch processing
settings_dict = {}
logs = []

for completion_mode in COMPLETION_MODE_LIST:
    for model in MODEL_LIST:
        for dataset_num, dataset_name in enumerate(DATASET_NAME_LIST):
            print("-" * 100)
            print(f"dataset: {dataset_name:<15}| model: {model:<15}| completion mode: {completion_mode:<15}")

            if model in OPENAI_MODELS:
                #  Generate batch input file
                if model in OPENAI_NON_REASONING_MODELS:
                    reasoning_effort = None
                    temperature = COMMON_TEMPERATURE
                elif model in OPENAI_REASONING_MODELS:
                    reasoning_effort = COMMON_REASONING_EFFORT
                    temperature = None
                else:
                    raise ValueError(f"Unsupported model: {model}")

                settings = generate_openai_batch_input_file(
                    dataset_name=dataset_name,
                    model=model,
                    completion_mode=completion_mode,
                    max_output_tokens=COMMON_MAX_OUTPUT_TOKENS,
                    reasoning_effort=reasoning_effort,
                    temperature=temperature,
                )

                # Read batch input file
                batch_input_filepath = BATCH_INPUT_DIR / f"batch_input_{dataset_name}_{model}_{completion_mode}.jsonl"
                with open(batch_input_filepath, "rb") as f:
                    lines = f.readlines()
                    n_samples_input = len(lines)
                print(f"{n_samples_input} rows")

                # Batch processing
                client_openai = OpenAI(api_key=OPENAI_API_KEY)
                batch_input_file = client_openai.files.create(file=open(batch_input_filepath, "rb"), purpose="batch")
                batch_input_file_id = batch_input_file.id
                batch = client_openai.batches.create(
                    input_file_id=batch_input_file.id,
                    endpoint="/v1/responses",
                    completion_window="24h",
                )
                batch_id = batch.id
                print(f"Batch ID: {batch_id}")

            elif model in ANTHROPIC_MODELS:
                # Generate batch requests
                if model in ANTHROPIC_MODELS:
                    reasoning_effort = None
                    temperature = COMMON_TEMPERATURE
                else:
                    raise ValueError(f"Unsupported model: {model}")

                requests, settings = generate_anthropic_requests(
                    dataset_name=dataset_name,
                    model=model,
                    completion_mode=completion_mode,
                    max_output_tokens=COMMON_MAX_OUTPUT_TOKENS,
                    temperature=temperature,
                )
                n_samples_input = len(requests)
                print(f"{n_samples_input} rows")

                # Batch processing
                client_anthropic = Anthropic(api_key=ANTHROPIC_API_KEY)
                batch_input_file_id = ""
                batch = client_anthropic.messages.batches.create(requests=requests)
                batch_id = batch.id
                print(f"Batch ID: {batch_id}")

            else:
                raise ValueError(f"Unsupported model: {model}")

            settings_dict[(dataset_name, model, completion_mode)] = settings

            log_dict = {
                "dataset_name": dataset_name,
                "model": model,
                "completion_mode": completion_mode,
                "max_output_tokens": COMMON_MAX_OUTPUT_TOKENS,
                "reasoning_effort": reasoning_effort,
                "temperature": temperature,
                "n_samples_input": n_samples_input,
                "batch_input_file_id": batch_input_file_id,
                "batch_id": batch_id,
            }

            logs.append(log_dict)

            if dataset_num > 0 and TEST_MODE:
                break

# Save settings and logs
with open(BATCH_INPUT_DIR / "batch_input_settings.pkl", "wb") as f:
    pickle.dump(settings_dict, f)

logs_df = pd.DataFrame(logs)
logs_df.to_csv(BATCH_INPUT_DIR / "batch_input_logs.csv", index=False)

## Check

In [ ]:
# Check
for completion_mode in COMPLETION_MODE_LIST:
    model = MODEL_LIST[0]
    dataset_name = DATASET_NAME_LIST[0]
    print("-" * 100)
    print(f"dataset: {dataset_name:<15}| model: {model:<15}| completion mode: {completion_mode:<15}")
    settings = settings_dict[(dataset_name, model, completion_mode)]
    for id, setting in settings.items():
        print("-" * 100)
        print(f"id: {id}")
        print("test_prefix:")
        print(setting["test_prefix"])
        print("test_suffix:")
        print(setting["test_suffix"])
        print("system_prompt:")
        print(setting["system_prompt"])
        for user_prompt in setting["user_prompts"]:
            print(user_prompt["role"])
            print(user_prompt["content"])


In [ ]:
display(logs_df)